<a href="https://colab.research.google.com/github/XTMay/ML_DL/blob/main/NLP/AdvancedTextCleaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import unicodedata
from typing import List , Optional

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import re
import unicodedata
from typing import List, Optional


class AdvancedTextCleaner:
    """
    高级文本清洗器
    支持多种清洗策略和自定义规则
    """

    def __init__(self):
        """初始化清洗器，预编译正则表达式提高效率"""
        self.patterns = {
            # URL 匹配（包括 http、https、ftp 等）
            "url": re.compile(
                r"http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F]{2}))+"
            ),
            # 邮箱地址匹配
            "email": re.compile(
                r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"
            ),
            # 电话号码匹配（中国手机号）
            "phone": re.compile(r"1[3-9]\d{9}"),
            # HTML 标签匹配
            "html": re.compile(r"<[^>]+>"),
            # 多个空白字符
            "extra_space": re.compile(r"\s+"),
            # 特殊字符（保留基本标点）
            "special_chars": re.compile(
                r"[^\w\s\u4e00-\u9fff，。！？；：\"'（）【】《》]"
            ),
            # 数字匹配
            "numbers": re.compile(r"\d+"),
            # 英文字符
            "english": re.compile(r"[a-zA-Z]+"),
        }

    def remove_urls(self, text: str) -> str:
        """移除 URL 链接"""
        return self.patterns["url"].sub("", text)

    def remove_emails(self, text: str) -> str:
        """移除邮箱地址"""
        return self.patterns["email"].sub("", text)

    def remove_phone_numbers(self, text: str) -> str:
        """移除电话号码"""
        return self.patterns["phone"].sub("", text)

    def remove_html_tags(self, text: str) -> str:
        """移除 HTML 标签"""
        return self.patterns["html"].sub("", text)

    def normalize_whitespace(self, text: str) -> str:
        """标准化空白字符"""
        return self.patterns["extra_space"].sub(" ", text).strip()

    def remove_special_chars(self, text: str, keep_chinese_punct: bool = True) -> str:
        """
        移除特殊字符
        Args:
            text: 输入文本
            keep_chinese_punct: 是否保留中文标点符号
        """
        if keep_chinese_punct:
            pattern = re.compile(r"[^\w\s\u4e00-\u9fff，。！？；：\"'（）【】《》]")
        else:
            pattern = self.patterns["special_chars"]
        return pattern.sub("", text)

    def remove_numbers(self, text: str) -> str:
        """移除数字"""
        return self.patterns["numbers"].sub("", text)

    def remove_english(self, text: str) -> str:
        """移除英文字符"""
        return self.patterns["english"].sub("", text)

    def normalize_unicode(self, text: str) -> str:
        """Unicode 标准化"""
        return unicodedata.normalize("NFKC", text)

    def filter_by_length(
        self, text: str, min_length: int = 2, max_length: int = 1000
    ) -> Optional[str]:
        """
        根据长度过滤文本
        Returns:
            过滤后的文本，如果不符合长度要求则返回 None
        """
        if min_length <= len(text) <= max_length:
            return text
        return None

    def clean_text(self, text: str, steps: List[str] = None) -> str:
        """
        综合文本清洗
        Args:
            text: 原始文本
            steps: 清洗步骤列表，默认执行所有步骤
        """
        if not text or not isinstance(text, str):
            return ""

        if steps is None:
            steps = [
                "unicode",
                "html",
                "url",
                "email",
                "phone",
                "special_chars",
                "numbers",
                "english",
                "whitespace",
            ]

        cleaned_text = text

        if "unicode" in steps:
            cleaned_text = self.normalize_unicode(cleaned_text)
        if "html" in steps:
            cleaned_text = self.remove_html_tags(cleaned_text)
        if "url" in steps:
            cleaned_text = self.remove_urls(cleaned_text)
        if "email" in steps:
            cleaned_text = self.remove_emails(cleaned_text)
        if "phone" in steps:
            cleaned_text = self.remove_phone_numbers(cleaned_text)
        if "special_chars" in steps:
            cleaned_text = self.remove_special_chars(cleaned_text)
        if "numbers" in steps:
            cleaned_text = self.remove_numbers(cleaned_text)
        if "english" in steps:
            cleaned_text = self.remove_english(cleaned_text)
        if "whitespace" in steps:
            cleaned_text = self.normalize_whitespace(cleaned_text)

        return cleaned_text

    def batch_clean(self, texts: List[str], steps: List[str] = None) -> List[str]:
        """批量清洗文本"""
        return [self.clean_text(text, steps) for text in texts]

    def get_cleaning_stats(
        self, original_texts: List[str], cleaned_texts: List[str]
    ) -> dict:
        """获取清洗统计信息"""
        original_lengths = [len(text) for text in original_texts]
        cleaned_lengths = [len(text) for text in cleaned_texts]

        return {
            "total_texts": len(original_texts),
            "avg_original_length": sum(original_lengths) / len(original_lengths),
            "avg_cleaned_length": sum(cleaned_lengths) / len(cleaned_lengths),
            "compression_ratio": sum(cleaned_lengths) / sum(original_lengths),
            "empty_after_cleaning": sum(1 for text in cleaned_texts if not text.strip()),
        }


In [ ]:
if __name__ == "__main__":
    cleaner = AdvancedTextCleaner()

    sample_texts = [
        "访问我们的网站 https://example.com ，联系邮箱：contact@example.com ！！！",
        "<p>这是HTML文本</p> ，包含<strong>标签</strong>",
        "我的电话号码是13812345678，请联系我。",
        "自然语言处理（NLP）是AI的重要分支！！！@#$%",
        "多余的   空格 需要处理",
    ]

    print("=== 文本清洗演示 ===")
    for i, text in enumerate(sample_texts):
        cleaned = cleaner.clean_text(text)
        print(f"\n文本 {i+1}:")
        print(f"原文：{text}")
        print(f"清洗后：{cleaned}")

    cleaned_texts = cleaner.batch_clean(sample_texts)
    stats = cleaner.get_cleaning_stats(sample_texts, cleaned_texts)

    print("\n=== 清洗统计信息 ===")
    for key, value in stats.items():
        print(f"{key}: {value:.3f}" if isinstance(value, float) else f"{key}: {value}")

=== 文本清洗演示 ===

文本 1:
原文：访问我们的网站 https://example.com ，联系邮箱：contact@example.com ！！！
清洗后：访问我们的网站 联系邮箱

文本 2:
原文：<p>这是HTML文本</p> ，包含<strong>标签</strong>
清洗后：这是文本 包含标签

文本 3:
原文：我的电话号码是13812345678，请联系我。
清洗后：我的电话号码是请联系我。

文本 4:
原文：自然语言处理（NLP）是AI的重要分支！！！@#$%
清洗后：自然语言处理是的重要分支

文本 5:
原文：多余的   空格 需要处理
清洗后：多余的 空格 需要处理

=== 清洗统计信息 ===
total_texts: 5
avg_original_length: 31.600
avg_cleaned_length: 11.200
compression_ratio: 0.354
empty_after_cleaning: 0
